# DL-04 · Stable Neural Training
### Section 04 — Make optimization observable, diagnosable, and reproducible

**Prerequisites:** Lessons DL-01 through DL-03. You should be able to write a PyTorch
training loop, derive a forward pass, explain backpropagation, and gradient-check
a small computation. **Estimated time:** 5–7 hours.


> **Canonical learner route · module DL-04 of 67**
>
> Required prerequisites: **DL-01, DL-02, DL-03** · Previous: **DL-03** ·
> Next after mastery: **DL-05** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

Diagnose underfitting, overfitting, vanishing/exploding gradients, dead units,
unstable loss, and nondeterminism; choose initialization, normalization,
optimizer, schedule, regularization, clipping, and early stopping based on evidence.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · DL-04 · Stable Neural Training

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |


## 2 · Historical Motivation

Deeper networks were theoretically expressive long before they trained reliably.
Xavier/He initialization, ReLU, normalization, residual connections, adaptive
optimizers, and accelerator-aware practice turned fragile demonstrations into
repeatable systems. Architecture and training procedure cannot be separated.


## 3 · Intuition and Visual Understanding

Training is a controlled feedback system. Activations flow forward, gradients
flow backward, the optimizer changes parameters, and validation measures whether
those changes generalize. Log distributions and norms—not only final accuracy.

```text
initialization → forward statistics → loss → gradient norms → update size
      ↑                                                        ↓
      └──── checkpoint / schedule / early stop ← validation ───┘
```


## 4 · Mathematical Foundations

For a ReLU layer with fan-in $d$, He initialization uses
$$W_{ij}\sim\mathcal N(0,2/d).$$
**Read aloud:** each weight is sampled with zero mean and variance two divided by
the number of inputs. For $d=100$, standard deviation is $\sqrt{0.02}\approx0.141$.
This approximately preserves signal variance through ReLU layers; it is not a
guarantee against poor data scaling or bad learning rates.

Gradient clipping rescales $g$ when $\lVert g\rVert_2>c$:
$$g\leftarrow g\min(1,c/\lVert g\rVert_2).$$
Here $c$ is the maximum norm. Clipping limits an update; it does not repair a
systematically broken model.


## 5 · Manual Implementation from Scratch

Compare activation variance under naive and He initialization. A stable network
keeps useful signal neither collapsing to zero nor exploding with depth.


In [ ]:
import random
import numpy as np
import torch
from torch import nn

torch.manual_seed(42)
x = torch.randn(512, 100)
naive = torch.randn(100, 100)
he = torch.randn(100, 100) * (2 / 100) ** 0.5
print("input var", round(x.var().item(), 3),
      "naive ReLU var", round(torch.relu(x @ naive).var().item(), 3),
      "He ReLU var", round(torch.relu(x @ he).var().item(), 3))


## 6 · Visualization

Plot training and validation loss together. Four common shapes matter: both high
(underfit), train down/validation up (overfit), both noisy/diverging (optimization
failure), and both falling then flattening (candidate convergence).


In [ ]:
history = {"train": [1.0, .72, .51, .37, .28, .22],
           "validation": [1.05, .78, .59, .54, .61, .73]}
best_epoch = int(np.argmin(history["validation"]))
print("best validation epoch", best_epoch,
      "restore it instead of the final epoch")


## 7 · Failure Modes and Common Mistakes

- Changing several controls at once, making diagnosis impossible.
- Selecting on the test set or reporting the last rather than best validation epoch.
- Using dropout or batch normalization in the wrong model mode.
- Treating gradient clipping as a cure for an excessive learning rate.
- Comparing optimizers with unequal budgets or unrecorded seeds.
- Reporting one lucky run as a robust improvement.


## 8 · Library Implementation

Build an explicit recipe: initialize layers, choose optimizer and schedule, log
losses/metrics/gradient norm, clip only when justified, save the best validation
checkpoint, and evaluate the test set once.


In [ ]:
model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 10))
for layer in model.modules():
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
print(model, optimizer.__class__.__name__, scheduler.__class__.__name__)


## 9 · Realistic Case Study

For handwritten digits, compare logistic regression, an MLP, and a CNN under the
same split. Run a controlled dropout ablation, preserve the best checkpoint, and
report mean/variation across seeds. The phase checkpoint implements this pattern.


## 10 · Production and Learning Considerations

Record seeds, deterministic settings, hardware, framework version, data/split
fingerprints, full configuration, selected epoch, and checkpoint hash. Exact
bitwise reproducibility can trade off against speed; declare the chosen level.


## 11 · Tradeoff Analysis

SGD often needs more tuning but can generalize strongly; AdamW is forgiving and
efficient. Batch normalization depends on batch statistics; layer normalization
does not. Dropout reduces co-adaptation but can slow fitting. Early stopping saves
compute but makes the stopping rule part of the model-selection procedure.


## 12 · Readiness and Interview Preparation

Given training/validation curves and gradient norms, identify the likely failure,
propose one controlled intervention, predict its effect, and define what evidence
would falsify the diagnosis.


## 13 · Teach-Back

Explain why good initialization matters even though training changes every weight.
Contrast normalization, weight decay, dropout, clipping, scheduling, and early
stopping: each addresses a different failure and they are not interchangeable.


## 14 · Exercises, Self-Check, and Solutions

1. **Guided (20 min):** calculate He standard deviation for fan-in 256.
   Self-check: $\sqrt{2/256}\approx0.0884$.
2. **Independent (40 min):** log global gradient norm and the update-to-weight
   ratio for every epoch. Flag non-finite values.
3. **Challenge (90 min):** compare SGD and AdamW across three fixed seeds under
   equal epoch budgets; report mean, range, curves, and practical effect size.

Readiness threshold: 80%, a correct diagnosis exercise, and no use of test data
for choosing optimizer, epoch, architecture, or regularization.


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **DL-04 · Stable Neural Training** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember DL-04 · Stable Neural Training: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · DL-04

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
